# 05 - Model Training (Full Protocol)

Trains all 4 models under **exactly the same protocol**: AdamW, cosine LR
schedule, mixed precision, gradient clipping, early stopping, seed=42,
batch size 8, 300 epochs (with early stopping so it may finish sooner).

| # | Model | Loss | Checkpoint |
|---|---|---|---|
| 1 | BaselineNet | Charbonnier | `results/checkpoints/BaselineNet.pth` |
| 2 | BaselineNet | Global Gen. Charbonnier (beta=0.845) | `results/checkpoints/Baseline_GenCharbonnier.pth` |
| 3 | DistributionMixtureRestorationNet (FiLM off) | Mixture NLL | `results/checkpoints/LDMH.pth` |
| 4 | DistributionMixtureRestorationNet (FiLM on) | Mixture NLL (Full model) | `results/checkpoints/DistributionMixtureRestorationNet.pth` |

Each model is trained fully independently (fresh weights, fresh optimizer).
Every checkpoint stores: model state dict, optimizer state dict, epoch,
best validation PSNR, best validation SSIM, and full training history.
CSV logs, TensorBoard logs, and loss/PSNR/SSIM curve PNGs are saved
automatically for every model.

**Requires:** `pip install lpips tensorboard` (run the cell below). LPIPS
downloads a small pretrained AlexNet backbone on first use -- needs
internet access once; if unavailable, LPIPS logs as NaN and everything
else still runs normally.

In [1]:
%pip install lpips tensorboard -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
print(sys.executable)

c:\Users\DELL\AppData\Local\Programs\Python\Python310\python.exe


In [4]:
!py -m pip install ipykernel

   ---------------------------------------- 0.0/5.4 MB ? eta -:--:--
   ------------------------- -------------- 3.4/5.4 MB 20.6 MB/s eta 0:00:01
   ---------------------------------------- 5.4/5.4 MB 20.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/626.0 kB ? eta -:--:--
   --------------------------------------- 626.0/626.0 kB 26.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/619.5 kB ? eta -:--:--
   --------------------------------------- 619.5/619.5 kB 29.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---------------------------------------- 4.9/4.9 MB 30.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 30.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [5]:
!py -m ipykernel install --user --name cuda-pytorch --display-name "Python (CUDA PyTorch)"

Installed kernelspec cuda-pytorch in C:\Users\DELL\AppData\Roaming\jupyter\kernels\cuda-pytorch


In [7]:
!py -m ipykernel install --user --name cuda-pytorch --display-name "Python (CUDA PyTorch)"

Installed kernelspec cuda-pytorch in C:\Users\DELL\AppData\Roaming\jupyter\kernels\cuda-pytorch


In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
import sys, os
sys.path.insert(0, "..")

import torch
from torch.utils.data import DataLoader

from utils.data import match_pairs, split_pairs, PairedRestorationDataset
from utils.train import train_model, set_seed
from models.restoration_net import BaselineNet, DistributionMixtureRestorationNet
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [ ]:

PROJECT_ROOT = Path("..").resolve()

TRAIN_GT_DIR = PROJECT_ROOT / "train" / "train" / "GT"
TRAIN_NOISY_DIR = PROJECT_ROOT / "train" / "train" / "NoisyLR"



SEED = 42
BATCH_SIZE = 8
EPOCHS = 200
PATIENCE = 30         
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
GLOBAL_BETA = 0.845     
VAL_FRAC = 0.1

CKPT_DIR = "../results/checkpoints"
LOG_DIR = "../results/logs"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

set_seed(SEED)


In [ ]:
all_pairs = match_pairs(TRAIN_GT_DIR, TRAIN_NOISY_DIR)   # full dataset, no cap
train_pairs, val_pairs = split_pairs(all_pairs, val_frac=VAL_FRAC, seed=SEED)
print(f"train: {len(train_pairs)}  val: {len(val_pairs)}")


train_ds = PairedRestorationDataset(train_pairs, augment=True)
val_ds = PairedRestorationDataset(val_pairs, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


train: 2880  val: 320


## Model 1/4 -- BaselineNet (Charbonnier)

In [5]:
set_seed(SEED)
model_baseline = BaselineNet(base_ch=32, n_lr_blocks=4, n_hr_blocks=2)

result_1 = train_model(
    model_baseline, "BaselineNet", train_loader, val_loader,
    model_type="plain", loss_type="charbonnier",
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM, patience=PATIENCE, seed=SEED,
    ckpt_dir=CKPT_DIR, log_dir=LOG_DIR, device=device,
)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\Owner/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [07:02<00:00, 578kB/s] 


Loading model from: c:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\alex.pth
[BaselineNet] epoch 1/200  train_loss=0.0433  val_loss=0.0408  PSNR=25.905  SSIM=0.6421  LPIPS=0.4567  lr=2.00e-04  t=215.7s
[BaselineNet] epoch 2/200  train_loss=0.0392  val_loss=0.0390  PSNR=26.238  SSIM=0.6732  LPIPS=0.4085  lr=2.00e-04  t=156.3s
[BaselineNet] epoch 3/200  train_loss=0.0382  val_loss=0.0379  PSNR=26.486  SSIM=0.6824  LPIPS=0.3859  lr=2.00e-04  t=155.5s
[BaselineNet] epoch 4/200  train_loss=0.0371  val_loss=0.0367  PSNR=26.785  SSIM=0.6952  LPIPS=0.3608  lr=2.00e-04  t=165.1s
[BaselineNet] epoch 5/200  train_loss=0.0360  val_loss=0.0357  PSNR=27.073  SSIM=0.7046  LPIPS=0.3409  lr=2.00e-04  t=137.2s
[BaselineNet] epoch 6/200  train_loss=0.0353  val_loss=0.0353  PSNR=27.149  SSIM=0.7137  LPIPS=0.3267  lr=2.00e-04  t=139.3s
[BaselineNet] epoch 7/200  train_loss=0.0349  val_loss=0.0349  PSNR=27.261  SSIM=0.7191  LPIPS=0.3289  lr=1.99e-04  t=164.0s
[Bas

## Model 2/4 -- BaselineNet + Global Generalized Charbonnier (beta=0.845)

In [6]:
set_seed(SEED)
model_gen_charbonnier = BaselineNet(base_ch=32, n_lr_blocks=4, n_hr_blocks=2)

result_2 = train_model(
    model_gen_charbonnier, "Baseline_GenCharbonnier", train_loader, val_loader,
    model_type="plain", loss_type="gen_charbonnier", global_beta=GLOBAL_BETA,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM, patience=PATIENCE, seed=SEED,
    ckpt_dir=CKPT_DIR, log_dir=LOG_DIR, device=device,
)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\alex.pth
[Baseline_GenCharbonnier] epoch 1/200  train_loss=0.0656  val_loss=0.0625  PSNR=25.903  SSIM=0.6424  LPIPS=0.4511  lr=2.00e-04  t=138.0s
[Baseline_GenCharbonnier] epoch 2/200  train_loss=0.0602  val_loss=0.0598  PSNR=26.287  SSIM=0.6738  LPIPS=0.4150  lr=2.00e-04  t=145.8s
[Baseline_GenCharbonnier] epoch 3/200  train_loss=0.0590  val_loss=0.0590  PSNR=26.432  SSIM=0.6786  LPIPS=0.3939  lr=2.00e-04  t=104.7s
[Baseline_GenCharbonnier] epoch 4/200  train_loss=0.0577  val_loss=0.0573  PSNR=26.705  SSIM=0.6910  LPIPS=0.3735  lr=2.00e-04  t=97.7s
[Baseline_GenCharbonnier] epoch 5/200  train_loss=0.0563  val_loss=0.0558  PSNR=26.994  SSIM=0.7002  LPIPS=0.3487  lr=2.00e-04  t=97.9s
[Baseline_GenCharbonnier] epoch 6/200  train_loss=0.0550  val_loss=0.0551  PSNR=27.109  SSIM=0.7116  LPIPS=0.3274  lr=2.00e-

## Model 3/4 -- Proposed DistributionMixtureRestorationNet (LDMH loss only, FiLM off)

In [6]:
set_seed(SEED)
model_ldmh = DistributionMixtureRestorationNet(base_ch=32, n_components=3,
                                                n_lr_blocks=4, n_hr_blocks=2, use_film=False)

result_3 = train_model(
    model_ldmh, "LDMH", train_loader, val_loader,
    model_type="ldmh", loss_type="mixture", aux_weight=1.0,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM, patience=PATIENCE, seed=SEED,
    ckpt_dir=CKPT_DIR, log_dir=LOG_DIR, device=device,
)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\alex.pth
[LDMH] epoch 1/200  train_loss=0.3735  val_loss=0.3391  PSNR=26.088  SSIM=0.6557  LPIPS=0.4288  lr=2.00e-04  t=71.7s
    [component check] beta=[0.404, 1.439, 2.407]  usage=[0.033, 0.382, 0.584]  min_pairwise_dist=0.968
[LDMH] epoch 2/200  train_loss=0.3104  val_loss=0.2829  PSNR=26.330  SSIM=0.6751  LPIPS=0.3982  lr=2.00e-04  t=65.8s
    [component check] beta=[0.41, 1.476, 2.414]  usage=[0.033, 0.384, 0.582]  min_pairwise_dist=0.938
[LDMH] epoch 3/200  train_loss=0.2545  val_loss=0.2269  PSNR=26.529  SSIM=0.6843  LPIPS=0.3836  lr=2.00e-04  t=65.8s
    [component check] beta=[0.416, 1.511, 2.42]  usage=[0.033, 0.375, 0.592]  min_pairwise_dist=0.908
[LDMH] epoch 4/200  train_loss=0.1983  val_loss=0.1701  PSNR=26.858  SSIM=0.6951  LPIPS=0.3589  lr=2.00e-04  t=66.0s
    [component check] beta=[0.424

## Model 4/4 -- Proposed DistributionMixtureRestorationNet + Adaptive Loss (Full Model, FiLM on)

In [5]:
set_seed(SEED)
model_full = DistributionMixtureRestorationNet(base_ch=32, n_components=3,
                                                n_lr_blocks=4, n_hr_blocks=2, use_film=True)

result_4 = train_model(
    model_full, "DistributionMixtureRestorationNet", train_loader, val_loader,
    model_type="ldmh", loss_type="mixture", aux_weight=1.0,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM, patience=PATIENCE, seed=SEED,
    ckpt_dir=CKPT_DIR, log_dir=LOG_DIR, device=device,
)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\alex.pth
[DistributionMixtureRestorationNet] epoch 1/200  train_loss=0.3730  val_loss=0.3386  PSNR=26.162  SSIM=0.6639  LPIPS=0.4021  lr=2.00e-04  t=93.7s
    [component check] beta=[0.404, 1.439, 2.407]  usage=[0.034, 0.379, 0.587]  min_pairwise_dist=0.968
[DistributionMixtureRestorationNet] epoch 2/200  train_loss=0.3097  val_loss=0.2811  PSNR=26.826  SSIM=0.6927  LPIPS=0.3578  lr=2.00e-04  t=91.1s
    [component check] beta=[0.41, 1.476, 2.414]  usage=[0.033, 0.382, 0.585]  min_pairwise_dist=0.938
[DistributionMixtureRestorationNet] epoch 3/200  train_loss=0.2524  val_loss=0.2247  PSNR=27.167  SSIM=0.7119  LPIPS=0.3364  lr=2.00e-04  t=93.0s
    [component check] beta=[0.416, 1.511, 2.42]  usage=[0.033, 0.371, 0.597]  min_pairwise_dist=0.908
[DistributionMixtureRestorationNet] epoch 4/200  train_loss=0.1964  val_loss=0.1687  PSNR=27.292  SSIM=0.7159  LPIPS=0.3327  lr=2.00e-0

## Final summary -- all 4 models

In [7]:
import os
import torch

CKPT_DIR = "../results/checkpoints"
MODEL_NAMES = ["BaselineNet", "Baseline_GenCharbonnier", "LDMH", "DistributionMixtureRestorationNet"]

all_histories = {}
for name in MODEL_NAMES:
    ckpt_path = os.path.join(CKPT_DIR, f"{name}.pth")
    if not os.path.exists(ckpt_path):
        print(f"WARNING: checkpoint not found for {name} -- skipping.")
        continue
    ckpt = torch.load(ckpt_path, map_location="cpu")
    all_histories[name] = ckpt["history"]

print(f"Loaded histories for: {list(all_histories.keys())}")

Loaded histories for: ['BaselineNet', 'Baseline_GenCharbonnier', 'LDMH', 'DistributionMixtureRestorationNet']


In [8]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, history in all_histories.items():
    axes[0].plot(history["train_loss"], label=name)
    axes[1].plot(history["val_psnr"], label=name)
    axes[2].plot(history["val_ssim"], label=name)

axes[0].set_title("Training Loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend(fontsize=8)

axes[1].set_title("Validation PSNR (accuracy)")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].legend(fontsize=8)

axes[2].set_title("Validation SSIM (accuracy)")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("SSIM")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../results/05_all_models_curves.png", dpi=150)
plt.show()
print("Saved: ../results/05_all_models_curves.png")

Saved: ../results/05_all_models_curves.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_8012\2075101305.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
for name, history in all_histories.items():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{name}: Loss")
    axes[0].legend()

    axes[1].plot(history["val_psnr"], color="green")
    axes[1].set_title(f"{name}: Val PSNR")

    axes[2].plot(history["val_ssim"], color="purple")
    axes[2].set_title(f"{name}: Val SSIM")

    plt.tight_layout()
    plt.savefig(f"../results/curves/{name}_final_curves.png", dpi=150)
    plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_8012\639473695.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
